# PARC2026 — CPU-only Dataset Prefetch to Google Drive

A100時間を動画downloadに使わないため、公開 `lerobot/libero_plus` v3 の **meta/data/videos** をCPU runtimeでGoogle Driveへ先に保存します。

このNotebookはGPU不要です。途中でtimeoutしても `hf_hub_download` のlocal metadataをDriveへ残すため、再実行でresumeできます。


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time
print('python:', sys.version)
print('GPU required: no')
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi: not present (CPU runtime is expected)')


## Google Drive mount
保存先は `MyDrive/parc2026-cache/datasets/lerobot_libero_plus_v3_train` です。20GB以上の空き容量を推奨します。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/parc2026-cache/datasets/lerobot_libero_plus_v3_train')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
usage=shutil.disk_usage('/content/drive/MyDrive')
print('Drive root:', DRIVE_ROOT)
print('Drive free GiB:', round(usage.free/2**30, 2))
if usage.free < 20 * 2**30:
    print('WARNING: 20GiB未満です。libero_plus全動画の保存に不足する可能性があります。')


## Revision pin + resumable download
現在の競技proxyで使っているrevisionをデフォルト固定します。必要なら `PI05_PUBLIC_DATASET_REVISION` 環境変数で上書きできます。


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.30'], check=True)
os.environ.setdefault('HF_HUB_DISABLE_XET','1')
from huggingface_hub import HfApi, hf_hub_download
DATASET_ID='lerobot/libero_plus'
PINNED_REVISION=os.environ.get('PI05_PUBLIC_DATASET_REVISION','f3f49f426d75030177b18778374005bc12ccd588')
api=HfApi()
info=api.dataset_info(DATASET_ID, revision=PINNED_REVISION)
REVISION=info.sha
files=api.list_repo_files(DATASET_ID, repo_type='dataset', revision=REVISION)
required=sorted(f for f in files if f.startswith('meta/') or (f.startswith('data/') and f.endswith('.parquet')) or (f.startswith('videos/') and f.endswith('.mp4')))
print('dataset:', DATASET_ID)
print('revision:', REVISION)
print('required files:', len(required))
for i,filename in enumerate(required,1):
    target=DRIVE_ROOT/filename
    if i==1 or i%10==0 or i==len(required):
        print(f'prefetch {i}/{len(required)}: {filename}')
    hf_hub_download(
        repo_id=DATASET_ID, repo_type='dataset', revision=REVISION,
        filename=filename, local_dir=str(DRIVE_ROOT),
    )


## Completion Gate
全required fileの存在とサイズを確認し、Driveへcompletion manifestを書きます。


In [ ]:
missing=[]
rows=[]
for filename in required:
    p=DRIVE_ROOT/filename
    if not p.exists() or p.stat().st_size <= 0:
        missing.append(filename)
    else:
        rows.append({'path':filename,'size_bytes':p.stat().st_size})
if missing:
    raise RuntimeError(f'missing/incomplete files: {missing[:10]} total={len(missing)}')
manifest={
    'schema_version':1,
    'dataset_id':DATASET_ID,
    'revision':REVISION,
    'required_file_count':len(required),
    'total_bytes':sum(r['size_bytes'] for r in rows),
    'files':rows,
    'completed_unix':int(time.time()),
}
marker=DRIVE_ROOT/'.parc_prefetch_complete.json'
marker.write_text(json.dumps(manifest, indent=2)+'\n')
print('total GiB:', round(manifest['total_bytes']/2**30, 2))
print('marker:', marker)
print('DRIVE PREFETCH GATE: PASS')


## 次
`DRIVE PREFETCH GATE: PASS` 後にruntimeをA100へ切り替え、`49_stage_training_dataset_from_drive.ipynb` でDrive→`/content`へlocal stagingしてからNotebook 50へ進みます。

Drive直読みで学習するとFUSE I/Oがボトルネックになりやすいため、学習時はlocal stagingを推奨します。
